In [1]:
# @title 🔧 Smart Setup — Run once per session( make sure to select runtime version as 2026.07)
# @markdown **First run (~20 min):** installs environments and saves reusable model weights to Drive.
# @markdown Later sessions reuse weights already present on Drive.
# @markdown If CondaColab restarts the runtime, reconnect and run this cell once more.

from IPython.utils import io
from IPython.display import HTML, display

print("⏳ Running setup and installations (this may take several minutes)...")

with io.capture_output() as captured:
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 1 — condacolab
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    try:
        import condacolab
        condacolab.check()
        print("✅ condacolab already active")
    except:
        print("⏳ Installing condacolab — runtime will restart automatically.")
        print("   Re-run this cell once after the restart.")
        !pip install -q condacolab
        import condacolab
        condacolab.install()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 2 — mamba (skip if already installed)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    import subprocess, shutil, os
    if shutil.which("mamba"):
        print("✅ mamba already installed")
    else:
        print("⏳ Installing mamba...")
        !conda install -c conda-forge mamba -y
        print("✅ mamba ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 3 — Mount Drive + create folder structure
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE    = '/content/drive/MyDrive/try_pipeline'
    DRIVE_WEIGHTS = f'{DRIVE_BASE}/weights'
    DRIVE_AF2     = f'{DRIVE_WEIGHTS}/colabfold_params'

    os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
    os.makedirs(DRIVE_AF2,     exist_ok=True)
    print("✅ Drive folders ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 4 — Clone or safely update the repository
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    REPO_DIR = '/content/batch_peptide_pipeline'
    REPO_URL = 'https://github.com/raghavagps/Alpha-Mod.git'

    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        status = subprocess.run(
            ['git', '-C', REPO_DIR, 'status', '--porcelain'],
            capture_output=True,
            text=True,
            check=True,
        )
        if status.stdout.strip():
            raise RuntimeError(
                "The existing Colab checkout contains local source changes. "
                "Start a fresh runtime before using the release notebook."
            )

        subprocess.run(
            ['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL],
            check=True,
        )
        update = subprocess.run(
            ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
            capture_output=True,
            text=True,
        )
        print(update.stdout)
        if update.returncode != 0:
            print(update.stderr)
            raise RuntimeError(
                "The Alpha-Mod checkout could not be updated safely. "
                "Start a fresh runtime and run this cell again."
            )
        print("✅ Alpha-Mod repository updated")
    elif os.path.exists(REPO_DIR):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout. "
            "Start a fresh runtime before continuing."
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
        print("✅ Alpha-Mod repository cloned")

    %cd /content/batch_peptide_pipeline
    !mkdir -p cache colabfold_params
    !rm -rf output

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 5 — Build environments (skip if already exist)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    def env_exists(name):
        r = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
        return name in r.stdout

    if env_exists('etflow_env'):
        print("✅ etflow_env already exists — skipping")
    else:
        print("⏳ Building etflow_env (~3 min)...")
        !mamba create -n etflow_env python=3.10 -y
        !mamba run -n etflow_env pip install requests rdkit biopython
        !mamba run -n etflow_env pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
        !mamba run -n etflow_env pip install torch-cluster==1.6.3 -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
        !mamba run -n etflow_env pip install etflow
        print("✅ etflow_env done")

    if env_exists('af2_env'):
        print("✅ af2_env already exists — skipping")
    else:
        print("⏳ Building af2_env (~3 min)...")
        !mamba create -n af2_env python=3.10 -y
        !mamba run -n af2_env pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
        !mamba run -n af2_env pip install --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
        print("✅ af2_env done")

    if env_exists('mace_env'):
        print("✅ mace_env already exists — skipping")
    else:
        print("⏳ Building mace_env (~2 min)...")
        !mamba create -n mace_env python=3.10 -y
        !mamba run -n mace_env pip install --upgrade torch torchvision torchaudio "numpy<2" --index-url https://download.pytorch.org/whl/cu118
        !mamba run -n mace_env pip install mace-torch ase openmm
        !mamba run -n mace_env pip install git+https://github.com/openmm/pdbfixer.git
        print("✅ mace_env done")

    !pip install biopython -q
    os.environ['MPLBACKEND'] = 'Agg'
    print("✅ All environments ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 6 — Weights (load from Drive if cached, else download + save)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    LOCAL_CACHE  = '/content/batch_peptide_pipeline/cache'
    LOCAL_AF2    = '/content/batch_peptide_pipeline/colabfold_params'
    ETFLOW_DRIVE = f'{DRIVE_WEIGHTS}/drugs-o3.ckpt'
    ETFLOW_LOCAL = f'{LOCAL_CACHE}/drugs-o3.ckpt'

    # -- ETFlow checkpoint --
    if os.path.exists(ETFLOW_DRIVE) and os.path.getsize(ETFLOW_DRIVE) > 50_000_000:
        print("\n⏳ ETFlow weights found on Drive — copying locally...")
        shutil.copy(ETFLOW_DRIVE, ETFLOW_LOCAL)
        print("✅ ETFlow weights ready (from Drive)")
    else:
        print("\n⏳ Downloading ETFlow weights from Zenodo (~101 MB)...")
        !wget --continue --tries=10 --timeout=30 -O {ETFLOW_LOCAL} "https://zenodo.org/records/14226681/files/drugs-o3.ckpt?download=1"
        print("💾 Saving ETFlow weights to Drive for future sessions...")
        shutil.copy(ETFLOW_LOCAL, ETFLOW_DRIVE)
        print("✅ ETFlow weights downloaded and saved to Drive")

    # -- AlphaFold params --
    COLABFOLD_CACHE = os.path.join(os.path.expanduser('~'), '.cache', 'colabfold', 'params')
    af2_drive_files = os.listdir(DRIVE_AF2) if os.path.exists(DRIVE_AF2) else []
    if len(af2_drive_files) > 5:
        print("\n⏳ AlphaFold params found on Drive — copying to cache (may take a few min)...")
        os.makedirs(COLABFOLD_CACHE, exist_ok=True)
        for fname in os.listdir(DRIVE_AF2):
            src = os.path.join(DRIVE_AF2, fname)
            dst = os.path.join(COLABFOLD_CACHE, fname)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
        print("✅ AlphaFold params ready (from Drive)")
    else:
        print("\n⏳ Downloading AlphaFold params (~1.8 GB)...")
        !mamba run -n af2_env python -m colabfold.download
        print("💾 Saving AlphaFold params to Drive for future sessions...")
        os.makedirs(DRIVE_AF2, exist_ok=True)
        if os.path.exists(COLABFOLD_CACHE):
            for fname in os.listdir(COLABFOLD_CACHE):
                src = os.path.join(COLABFOLD_CACHE, fname)
                dst = os.path.join(DRIVE_AF2, fname)
                if not os.path.exists(dst):
                    shutil.copy(src, dst)
            print("✅ AlphaFold params downloaded and saved to Drive")
        else:
            print("⚠️ Cache folder not found after download — params may not have saved to Drive")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 7 — Verify repository files
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    required_files = [
        '01_parse_input.py',
        '02_run_backbone.py',
        '03_run_sidechains.py',
        '04_stitch.py',
        '05_minimize.py',
        'main.py',
        'modifications.json',
        'residue_metadata.py',
    ]
    missing_files = [
        name for name in required_files
        if not os.path.isfile(os.path.join(REPO_DIR, name))
    ]
    if missing_files:
        raise RuntimeError(
            f"The cloned repository is missing required files: {missing_files}"
        )
    print("✅ Required Alpha-Mod scripts verified")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 8 — Warm MACE cache before parallel prediction
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("\n⏳ Loading the MACE-OFF medium model once on CPU...")
    print("   The first run may download approximately 17.5 MB.")

    mace_warmup = "\n".join([
        "from mace.calculators import mace_off",
        "mace_off(model='medium', device='cpu', default_dtype='float64')",
        "print('MACE_MODEL_READY')",
    ])

    mace_result = subprocess.run(
        [
            'conda', 'run', '--no-capture-output',
            '-n', 'mace_env', 'python', '-c', mace_warmup,
        ],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    print(mace_result.stdout)
    if mace_result.returncode != 0:
        print(mace_result.stderr)
        raise RuntimeError(
            "MACE model warm-up failed. Do not start parallel prediction."
        )
    if 'MACE_MODEL_READY' not in mace_result.stdout:
        raise RuntimeError("MACE warm-up finished without a readiness marker.")
    print("✅ MACE model is cached and loadable")

    print("\n🎉 Setup complete! Run Cell 2 for single sequence or Cell 3 for batch.")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 9 — Display setup log
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
display(HTML(f"""
<details>
    <summary style="cursor: pointer; color: #1a73e8; font-weight: bold; padding: 10px; border: 1px solid #1a73e8; border-radius: 5px; width: fit-content;">
        ✅ Setup Complete. Click here to view detailed installation logs.
    </summary>
    <div style="background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; margin-top: 10px; max-height: 500px; overflow-y: auto;">
        <pre style="white-space: pre-wrap; word-wrap: break-word;">{captured.stdout}</pre>
        <pre style="color: red; white-space: pre-wrap; word-wrap: break-word;">{captured.stderr}</pre>
    </div>
</details>
"""))

⏳ Running setup and installations (this may take several minutes)...


In [2]:
# @title ▶️ Single Sequence Prediction
# @markdown Enter a sequence and a filesystem-safe job name, then run this cell.
# @markdown Supported inputs include canonical sequences, known CCD codes in
# @markdown parentheses, MAP notation, and `|SMILES,ParentLetter|` novel residues.

sequence = "KET(AIB)AAKFERQHLDS"  # @param {type:"string"}
job_name = "example_sequence"  # @param {type:"string"}
# @markdown Final-only mode includes the final PDB and its three residue identity files.
download_mode = "final_pdb_only"  # @param ["final_pdb_only", "all_files"]

import html
import json
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

from google.colab import files
from IPython.display import HTML, display

def _final_bundle_files(folder):
    """Collect the final PDB and the identity files from one saved job."""
    import hashlib

    names = (
        "final_minimized.pdb",
        "residue_identity.csv",
        "modifications.metadata.json",
        "modifications.txt",
    )
    missing = [name for name in names if not (folder / name).is_file()]
    if missing:
        raise RuntimeError(f"{folder}: missing {', '.join(missing)}. Check the saved job before exporting.")
    metadata = json.loads((folder / "modifications.metadata.json").read_text(encoding="utf-8"))
    checksum = hashlib.sha256((folder / "modifications.txt").read_bytes()).hexdigest()
    if metadata.get("modifications_sha256") != checksum:
        raise RuntimeError(f"{folder}: modifications.txt and its metadata do not match.")
    return [folder / name for name in names]


REPO_DIR = '/content/batch_peptide_pipeline'
DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)

job_name = job_name.strip()
if not job_name or Path(job_name).name != job_name or job_name in {'.', '..'}:
    raise ValueError(
        "job_name must be a non-empty filesystem-safe name without slashes."
    )

for folder in ['converged', 'max_steps', 'failure_logs']:
    (drive_path / folder).mkdir(parents=True, exist_ok=True)

print(f"🚀 Starting job : {job_name}")
print(f"🧬 Sequence     : {sequence}\n")
print("⏳ Running pipeline...")

clean_env = {**os.environ, 'MPLBACKEND': 'Agg'}
result = subprocess.run(
    [
        'python', 'main.py',
        '--sequence', sequence,
        '--json', 'modifications.json',
        '--job_name', job_name,
    ],
    cwd=REPO_DIR,
    env=clean_env,
    capture_output=True,
    text=True,
)

full_log = (
    (result.stdout or '')
    + ('\n' if result.stdout and result.stderr else '')
    + (result.stderr or '')
)
if not full_log.strip():
    full_log = 'No subprocess output was captured.\n'

display(HTML(f"""
<details>
  <summary style="cursor:pointer;color:#1a73e8;font-weight:bold;padding:10px;border:1px solid #1a73e8;border-radius:5px;width:fit-content;">
    📊 Click here to view detailed pipeline logs
  </summary>
  <div style="background:#f8f9fa;padding:10px;border:1px solid #ddd;margin-top:10px;max-height:400px;overflow:auto;">
    <pre style="white-space:pre-wrap;word-wrap:break-word;">{html.escape(full_log)}</pre>
  </div>
</details>
"""))

if result.returncode != 0:
    diagnostic_dir = drive_path / 'failure_logs' / job_name
    diagnostic_dir.mkdir(parents=True, exist_ok=True)
    log_path = diagnostic_dir / 'pipeline.log'
    log_path.write_text(full_log, encoding='utf-8')

    useful_lines = [line.strip() for line in full_log.splitlines() if line.strip()]
    error_message = useful_lines[-1] if useful_lines else (
        f"Pipeline exited with code {result.returncode}."
    )
    failure_status = {
        'name': job_name,
        'folder': 'failed',
        'converged': False,
        'exit_code': int(result.returncode),
        'error_message': error_message,
        'diagnostic_log': str(log_path),
    }
    (diagnostic_dir / 'failure_status.json').write_text(
        json.dumps(failure_status, indent=2),
        encoding='utf-8',
    )

    failure_zip = f'/content/{job_name}_failure.zip'
    with zipfile.ZipFile(failure_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in diagnostic_dir.iterdir():
            if path.is_file():
                archive.write(path, f'failure_logs/{job_name}/{path.name}')

    print(f"\n❌ Job '{job_name}' failed (exit code {result.returncode}).")
    print(f"   {error_message}")
    print("⬇️ Downloading failure diagnostics...")
    files.download(failure_zip)

else:
    print(f"\n✅ Job '{job_name}' completed without a hard pipeline error.")
    time.sleep(3)

    job_folder = None
    status_folder = None
    for folder in ['converged', 'max_steps']:
        candidate = drive_path / folder / job_name
        if candidate.exists():
            job_folder = candidate
            status_folder = folder
            break

    if job_folder is None:
        print(
            "⚠️ The pipeline returned success, but its Drive output folder "
            "was not found. Inspect the detailed log."
        )
    else:
        print(f"📂 Saved under: {status_folder}/{job_name}/")

        if download_mode == 'final_pdb_only':
            pdb_source = job_folder / 'final_minimized.pdb'
            if not pdb_source.exists():
                print("⚠️ final_minimized.pdb was not found in the saved job folder.")
            else:
                bundle_files = _final_bundle_files(job_folder)
                zip_path = f'/content/{job_name}_final_with_identity.zip'
                with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
                    for path in bundle_files:
                        archive.write(path, f'{job_name}/{path.name}')
                print(f"⬇️ Downloading {job_name}_final_with_identity.zip...")
                files.download(zip_path)
        else:
            zip_path = f'/content/{job_name}_output.zip'
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
                for root, _, filenames in os.walk(job_folder):
                    for filename in filenames:
                        path = os.path.join(root, filename)
                        archive.write(
                            path,
                            os.path.relpath(path, str(job_folder.parent)),
                        )
            print(f"⬇️ Downloading {job_name}_output.zip...")
            files.download(zip_path)


🚀 Starting job : example_sequence
🧬 Sequence     : KET(AIB)AAKFERQHLDS

⏳ Running pipeline...



✅ Job 'example_sequence' completed without a hard pipeline error.
📂 Saved under: converged/example_sequence/
⬇️ Downloading example_sequence_final_with_identity.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# @title ▶️ Batch Prediction from FASTA (2 sequences in parallel)
# @markdown ⚠️  **This cell wipes all previous prediction output from Drive before starting.**
# @markdown Upload your .fasta file when prompted.
# @markdown Output + CSV will be auto-downloaded as a zip when done.

import html, os, shutil, json, zipfile, subprocess
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from google.colab import files
from IPython.utils import io
from IPython.display import HTML, display

def _final_bundle_files(folder):
    """Collect the final PDB and the identity files from one saved job."""
    import hashlib

    names = (
        "final_minimized.pdb",
        "residue_identity.csv",
        "modifications.metadata.json",
        "modifications.txt",
    )
    missing = [name for name in names if not (folder / name).is_file()]
    if missing:
        raise RuntimeError(f"{folder}: missing {', '.join(missing)}. Check the saved job before exporting.")
    metadata = json.loads((folder / "modifications.metadata.json").read_text(encoding="utf-8"))
    checksum = hashlib.sha256((folder / "modifications.txt").read_bytes()).hexdigest()
    if metadata.get("modifications_sha256") != checksum:
        raise RuntimeError(f"{folder}: modifications.txt and its metadata do not match.")
    return [folder / name for name in names]


REPO_DIR   = '/content/batch_peptide_pipeline'
DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)

# @markdown Final-only mode includes the final PDB and its three residue identity files.
download_mode = "final_pdb_only"  # @param ["final_pdb_only", "all_files"]

%cd /content/batch_peptide_pipeline

# ── FIX: override Colab's matplotlib_inline backend so conda envs don't crash ──
_clean_env = {**os.environ, 'MPLBACKEND': 'Agg'}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 — Wipe previous output (Keep visible for safety)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("🧹 Clearing previous prediction output from Drive...")
for folder in ['converged', 'max_steps', 'failure_logs']:
    p = drive_path / folder
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(exist_ok=True)

csv_path = drive_path / 'batch_summary.csv'
if csv_path.exists():
    csv_path.unlink()
print("✅ Previous output cleared\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 — Upload and parse FASTA (Must be visible)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("👇 Please upload your .fasta file below:")
uploaded = files.upload()

if len(uploaded) == 0:
    print("\n❌ No file uploaded. Please re-run this cell.")
else:
    fasta_file = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {fasta_file}")

    sequences = {}
    with open(fasta_file) as f:
        current_id = None
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                current_id = line[1:].strip()
            elif current_id and line:
                sequences[current_id] = sequences.get(current_id, '') + line

    print(f"Found {len(sequences)} sequences. Running 2 in parallel...")
    print("⏳ Processing pipeline... (Check dropdown below for progress)")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 3 & 4 — Run in parallel (CAPTURED)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    def save_failure_diagnostics(seq_id, log_text, exit_code, error_message):
        diagnostic_dir = drive_path / 'failure_logs' / seq_id
        diagnostic_dir.mkdir(parents=True, exist_ok=True)

        log_path = diagnostic_dir / 'pipeline.log'
        log_path.write_text(
            log_text if log_text.strip() else "No subprocess output was captured.\n",
            encoding='utf-8',
        )

        failure_status = {
            'name': seq_id,
            'folder': 'failed',
            'converged': False,
            'exit_code': int(exit_code),
            'error_message': error_message,
            'diagnostic_log': str(log_path),
        }

        with open(diagnostic_dir / 'failure_status.json', 'w') as handle:
            json.dump(failure_status, handle, indent=2)

        return failure_status


    def run_one_sequence(seq_id, sequence):
        try:
            cmd = [
                'python', 'main.py',
                '--sequence', sequence,
                '--json', 'modifications.json',
                '--job_name', seq_id,
            ]

            result = subprocess.run(
                cmd,
                cwd=REPO_DIR,
                env=_clean_env,
                capture_output=True,
                text=True,
            )

            full_log = (
                (result.stdout or '')
                + ('\n' if result.stdout and result.stderr else '')
                + (result.stderr or '')
            )

            if result.returncode == 0:
                print(f"  ✅ {seq_id} — Done")
                return True

            useful_lines = [line.strip() for line in full_log.splitlines() if line.strip()]
            error_message = useful_lines[-1] if useful_lines else (
                f"Pipeline exited with code {result.returncode}; see pipeline.log."
            )

            failure_status = save_failure_diagnostics(
                seq_id,
                full_log,
                result.returncode,
                error_message,
            )
            print(
                f"  ❌ {seq_id} — Failed (exit code {result.returncode}): "
                f"{failure_status['error_message']}"
            )
            print(f"     Diagnostic log: {failure_status['diagnostic_log']}")
            return False

        except Exception as exc:
            failure_status = save_failure_diagnostics(
                seq_id,
                f"Batch launcher error: {exc}\n",
                1,
                str(exc),
            )
            print(f"  ❌ {seq_id} — Launcher error: {failure_status['error_message']}")
            print(f"     Diagnostic log: {failure_status['diagnostic_log']}")
            return False

    # Capture the messy parallel logs
    with io.capture_output() as captured:
        print("=" * 60)
        print("  Running pipeline...")
        print("=" * 60)
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = {seq_id: executor.submit(run_one_sequence, seq_id, seq)
                       for seq_id, seq in sequences.items()}
        job_success = {seq_id: future.result() for seq_id, future in futures.items()}

    # Display the logs dropdown
    display(HTML(f"""
    <details>
        <summary style="cursor: pointer; color: #1a73e8; font-weight: bold; padding: 10px; border: 1px solid #1a73e8; border-radius: 5px; width: fit-content;">
            📊 View Parallel Processing Logs
        </summary>
        <div style="background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; margin-top: 10px; max-height: 300px; overflow-y: auto;">
            <pre style="white-space: pre-wrap; word-wrap: break-word;">{html.escape(captured.stdout)}</pre>
        </div>
    </details>
    """))

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 5 & 6 — Collect Results & Save CSV (Visible)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    all_results = []
    for seq_id in sequences.keys():
        status_json = None
        job_folder = None

        for subfolder in ['converged', 'max_steps']:
            candidate = drive_path / subfolder / seq_id / 'minimize_status.json'
            if candidate.exists():
                status_json = candidate
                job_folder = subfolder
                break

        failure_json = (
            drive_path / 'failure_logs' / seq_id / 'failure_status.json'
        )

        if status_json:
            with open(status_json) as f:
                status = json.load(f)

            status.pop('exploded', None)
            status.pop('explosion_msg', None)
            status['name'] = seq_id
            status['folder'] = job_folder
            all_results.append(status)

        elif failure_json.exists():
            with open(failure_json) as f:
                failure_status = json.load(f)

            failure_status['name'] = seq_id
            failure_status['folder'] = 'failed'
            failure_status['converged'] = False
            all_results.append(failure_status)

        else:
            all_results.append({
                'name': seq_id,
                'folder': 'failed',
                'converged': False,
                'error_message': 'No result status or failure diagnostic was found.',
            })

    if all_results:
        df = pd.DataFrame(all_results)
        df.to_csv(str(csv_path), index=False)
        n_conv = int(df['converged'].sum())
        n_max = int((df['folder'] == 'max_steps').sum())
        n_failed = int((df['folder'] == 'failed').sum())

        print(f"\n{'=' * 60}")
        print(
            f"  BATCH COMPLETE — ✅ {n_conv} Converged | "
            f"⚠️ {n_max} Max steps | ❌ {n_failed} Failed"
        )
        print(f"{'=' * 60}")
        print(f"📄 Summary saved to: {csv_path}")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 7 — Zip & Download (Visible)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    zip_path = ('/content/batch_final_with_identity.zip'
                if download_mode == 'final_pdb_only' else '/content/batch_output.zip')
    print(f"\n📦 Zipping output for download...")

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for subfolder in ['converged', 'max_steps', 'failure_logs']:
            p = drive_path / subfolder
            if p.exists():
                for job_dir in p.iterdir():
                    if not job_dir.is_dir(): continue
                    if download_mode == "final_pdb_only":
                        if (job_dir / 'final_minimized.pdb').is_file():
                            for path in _final_bundle_files(job_dir):
                                zf.write(path, f"{subfolder}/{job_dir.name}/{path.name}")
                    else:
                        for root, dirs, filenames in os.walk(job_dir):
                            for fname in filenames:
                                fpath = os.path.join(root, fname)
                                zf.write(fpath, os.path.relpath(fpath, str(drive_path)))
        if csv_path.exists():
            zf.write(str(csv_path), 'batch_summary.csv')

    print(f"⬇️  Downloading {Path(zip_path).name} ({download_mode})...")
    files.download(zip_path)
    print("\n🎉 Done!")

/content/batch_peptide_pipeline
🧹 Clearing previous prediction output from Drive...
✅ Previous output cleared

👇 Please upload your .fasta file below:


Saving input.fasta to input.fasta

✅ Uploaded: input.fasta
Found 3 sequences. Running 2 in parallel...
⏳ Processing pipeline... (Check dropdown below for progress)



  BATCH COMPLETE — ✅ 3 Converged | ⚠️ 0 Max steps | ❌ 0 Failed
📄 Summary saved to: /content/drive/MyDrive/try_pipeline/batch_summary.csv

📦 Zipping output for download...
⬇️  Downloading batch_final_with_identity.zip (final_pdb_only)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Done!


In [ ]:
# @title 🔬 Visualize Final Minimized Structure(s)
# @markdown Scans Drive for every completed job (works after either the single-sequence
# @markdown or batch cell) and renders each `final_minimized.pdb` in 3D — backbone as
# @markdown cartoon, modified/NCAA residues highlighted as colored sticks with labels.
# @markdown Safe to re-run any time; does not modify anything on Drive.
import sys
!{sys.executable} -m pip install py3Dmol -q

import json
import py3Dmol

from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)

# Same reference list used in 05_minimize.py — anything NOT in here is treated
# as a modified / non-canonical residue and gets highlighted.
STANDARD_RESIDUES = {
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY',
    'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER',
    'THR', 'TRP', 'TYR', 'VAL',
    'HID', 'HIE', 'HIP', 'CYX', 'ASH', 'GLH',
    'ACE', 'NME', 'NMA',
}


def find_ncaa_residues(pdb_path):
    """
    Scan a PDB file and return {resnum: (resname, ca_xyz)} for every residue
    whose name is NOT in STANDARD_RESIDUES. ca_xyz falls back to the first
    atom's coordinates if no CA atom is present (some NCAAs may lack one).
    """
    ncaa = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith(('ATOM', 'HETATM')):
                continue
            resname = line[17:20].strip()
            resnum = line[22:26].strip()
            atom_name = line[12:16].strip()
            if resname in STANDARD_RESIDUES:
                continue
            xyz = (float(line[30:38]), float(line[38:46]), float(line[46:54]))
            if resnum not in ncaa:
                ncaa[resnum] = [resname, xyz]  # default to first atom seen
            elif atom_name == 'CA':
                ncaa[resnum][1] = xyz  # prefer CA if we find it later
    return ncaa


def visualize_pdb(pdb_path, title, status=None):
    with open(pdb_path) as f:
        pdb_data = f.read()

    ncaa = find_ncaa_residues(pdb_path)

    view = py3Dmol.view(width=750, height=500)
    view.addModel(pdb_data, 'pdb')

    # Whole chain as a cartoon so the fold is legible.
    view.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    # Highlight each NCAA residue as thick sticks + a text label.
    for resnum, (resname, xyz) in ncaa.items():
        view.addStyle({'resi': resnum}, {'stick': {'color': 'orange', 'radius': 0.3}})
        view.addLabel(f"{resname} {resnum}", {
            'position': {'x': xyz[0], 'y': xyz[1], 'z': xyz[2]},
            'backgroundColor': 'orange',
            'backgroundOpacity': 0.8,
            'fontColor': 'black',
            'fontSize': 12,
        })

    view.zoomTo()

    print(f"\n📌 {title}")
    if status:
        conv_word = (
            "✅ Converged" if status.get('converged')
            else "⚠️ Max steps reached"
        )
        rmsd = status.get('rmsd_heavy')
        rmsd_txt = f", heavy-atom RMSD from pre-minimization = {rmsd:.3f} Å" if rmsd is not None else ""
        print(f"   {conv_word}{rmsd_txt}")
    if ncaa:
        tags = ', '.join(f"{resname}{resnum}" for resnum, (resname, _) in sorted(ncaa.items(), key=lambda x: int(x[0])))
        print(f"   Modified residues (orange): {tags}")
    else:
        print("   No non-canonical residues detected in this structure.")

    view.show()


# ── Discover every completed job on Drive (single-sequence and/or batch runs) ──
jobs_found = []
for status_folder in ['converged', 'max_steps']:
    folder = drive_path / status_folder
    if not folder.exists():
        continue
    for job_dir in sorted(folder.iterdir()):
        pdb_file = job_dir / 'final_minimized.pdb'
        if pdb_file.exists():
            jobs_found.append((job_dir.name, status_folder, pdb_file))

if not jobs_found:
    print("❌ No final_minimized.pdb files found on Drive yet. Run a prediction first (Cell 2 or Cell 3).")
else:
    print(f"🔍 Found {len(jobs_found)} structure(s) to visualize.")
    for job_name, status_folder, pdb_file in jobs_found:
        status = None
        status_file = pdb_file.parent / 'minimize_status.json'
        if status_file.exists():
            with open(status_file) as f:
                status = json.load(f)
        visualize_pdb(pdb_file, f"{job_name}  [{status_folder}]", status=status)

In [ ]:
# @title 🛟 Recovery — Run only if a batch stopped midway
# @markdown Scans Drive for converged, max-steps, and failed jobs; rebuilds
# @markdown `batch_summary.csv`; and downloads the recovered files.
# @markdown This cell does not delete or rerun any prediction.

import json
import os
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import files

DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)
csv_path = drive_path / 'batch_summary.csv'

print("🔍 Scanning Drive for recoverable jobs...")
all_results = []

for folder in ['converged', 'max_steps']:
    folder_path = drive_path / folder
    if not folder_path.exists():
        continue

    for job_dir in sorted(folder_path.iterdir()):
        if not job_dir.is_dir():
            continue
        status_file = job_dir / 'minimize_status.json'
        if not status_file.exists():
            continue

        with status_file.open(encoding='utf-8') as handle:
            status = json.load(handle)

        status.pop('exploded', None)
        status.pop('explosion_msg', None)
        status['name'] = job_dir.name
        status['folder'] = folder
        all_results.append(status)

        if status.get('converged'):
            print(
                f"  ✅ {job_dir.name}: converged in "
                f"{status.get('steps', '?')} steps"
            )
        else:
            print(
                f"  ⚠️ {job_dir.name}: reached "
                f"{status.get('steps', '?')} steps"
            )

failure_root = drive_path / 'failure_logs'
if failure_root.exists():
    for job_dir in sorted(failure_root.iterdir()):
        if not job_dir.is_dir():
            continue
        failure_file = job_dir / 'failure_status.json'
        if not failure_file.exists():
            continue

        with failure_file.open(encoding='utf-8') as handle:
            failure = json.load(handle)

        failure['name'] = job_dir.name
        failure['folder'] = 'failed'
        failure['converged'] = False
        all_results.append(failure)
        print(f"  ❌ {job_dir.name}: hard failure diagnostics recovered")

if not all_results:
    print("❌ No completed jobs or failure diagnostics were found on Drive.")
else:
    dataframe = pd.DataFrame(all_results)
    preferred_columns = [
        'name', 'folder', 'converged', 'n_atoms', 'steps',
        'e0', 'e1', 'delta_e', 'time_sec', 'time_per_step',
        'rmsd_all', 'rmsd_heavy', 'max_disp',
        'exit_code', 'error_message', 'diagnostic_log',
    ]
    ordered = [column for column in preferred_columns if column in dataframe.columns]
    ordered += [column for column in dataframe.columns if column not in ordered]
    dataframe = dataframe[ordered]
    dataframe.to_csv(csv_path, index=False)

    n_converged = int(dataframe['converged'].fillna(False).sum())
    n_max_steps = int((dataframe['folder'] == 'max_steps').sum())
    n_failed = int((dataframe['folder'] == 'failed').sum())

    print("\n" + "=" * 60)
    print(f"  RECOVERED — {len(dataframe)} job record(s)")
    print("=" * 60)
    print(f"  ✅ Converged : {n_converged}")
    print(f"  ⚠️ Max steps : {n_max_steps}")
    print(f"  ❌ Failed    : {n_failed}")
    print(f"  📄 CSV       : {csv_path}")

    zip_path = '/content/batch_recovered.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for folder in ['converged', 'max_steps', 'failure_logs']:
            folder_path = drive_path / folder
            if not folder_path.exists():
                continue
            for root, _, filenames in os.walk(folder_path):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    archive.write(path, os.path.relpath(path, str(drive_path)))
        archive.write(str(csv_path), 'batch_summary.csv')

    print("⬇️ Downloading batch_recovered.zip...")
    files.download(zip_path)
